# 02 — Tách Silver và Rejected từ Bronze

### 1. Xem dữ liệu trước khi xử lý

Notebook 01 tạo bảng `delivery_bronze`. Hai cột thời gian đang là chuỗi
dạng `MM-DD HH:MM:SS`.

In [ ]:
import pyspark.sql.functions as F
from lakehouse.common import haversine, parse_timestamp, require_unique_orders, valid_coordinates
from lakehouse.notebooks import load_notebook_steps

shared = load_notebook_steps()
BRONZE = shared.BRONZE
REJECTED = shared.REJECTED
SILVER = shared.SILVER
reconcile_order_keys = shared.reconcile_order_keys

spark = shared.get_spark("Delivery - Silver")

In [ ]:
bronze = spark.read.format("delta").load(BRONZE)
bronze.select("order_id", "accept_time", "delivery_time").show(5, truncate=False)

### 2. Áp dụng điều kiện Silver

1. Ép kiểu mã và tọa độ. Gắn năm 2022 vào thời gian nhận, giao.
2. Tính `delivery_minutes = (delivery_ts - accept_ts) / 60` với hiệu thời gian tính bằng giây.
3. Giữ đơn có hai timestamp hợp lệ và thời lượng dương. Đưa đơn còn lại vào Rejected
   với lý do `invalid_timestamp` hoặc `non_positive_duration`.
4. Gắn cờ GPS, qua ngày, thời lượng trên 24 giờ và khoảng cách trên 50 km.
   Hai ngưỡng 24 giờ và 50 km dùng để chọn tập học Spark ở notebook 04.

`transform_silver()` gọi `split_silver()`, ghi hai bảng và đối soát
mỗi khóa Bronze xuất hiện đúng một lần trong hợp Silver và Rejected.

In [ ]:
def split_silver(df):
    d = (df.withColumn("accept_ts", parse_timestamp("accept_time"))
           .withColumn("delivery_ts", parse_timestamp("delivery_time")))
    for c in ["lng", "lat", "accept_gps_lng", "accept_gps_lat", "delivery_gps_lng", "delivery_gps_lat"]:
        d = d.withColumn(c, F.col(c).cast("double"))
    for c in ["region_id", "courier_id", "aoi_id", "aoi_type"]:
        d = d.withColumn(c, F.col(c).cast("long"))
    d = (d.withColumn("delivery_minutes",
                      (F.col("delivery_ts").cast("long") - F.col("accept_ts").cast("long")) / 60.0)
         .withColumn("day_gap", F.datediff(F.to_date("delivery_ts"), F.to_date("accept_ts")))
         .withColumn("distance_km", haversine("lat", "lng", "accept_gps_lat", "accept_gps_lng"))
         .withColumn("delivery_stop_distance_km",
                     haversine("lat", "lng", "delivery_gps_lat", "delivery_gps_lng")))
    d = (d.withColumn("q_invalid_timestamp", F.col("accept_ts").isNull() | F.col("delivery_ts").isNull())
         .withColumn("q_zero_duration", F.col("delivery_minutes") == 0)
         .withColumn("q_negative_duration", F.col("delivery_minutes") < 0)
         .withColumn("q_cross_day", F.col("day_gap") > 0)
         .withColumn("q_duration_gt24h", F.col("delivery_minutes") > 1440)
         .withColumn("q_accept_gps_invalid", ~valid_coordinates("accept_gps_lat", "accept_gps_lng"))
         .withColumn("q_delivery_gps_invalid", ~valid_coordinates("delivery_gps_lat", "delivery_gps_lng"))
         .withColumn("q_accept_distance_gt50km", F.col("distance_km") > 50)
         .withColumn("q_delivery_distance_gt50km", F.col("delivery_stop_distance_km") > 50)
         .withColumn("eligible_duration", F.col("accept_ts").isNotNull() & F.col("delivery_ts").isNotNull()
                     & (F.col("delivery_minutes") > 0))
         .withColumn("eligible_ml_distance", F.col("eligible_duration") & F.col("distance_km").isNotNull()
                     & ~F.col("q_accept_gps_invalid")))
    rejected = (d.filter(~F.col("eligible_duration"))
                .withColumn("reject_reason",
                            F.when(F.col("q_invalid_timestamp"), "invalid_timestamp")
                             .when(F.col("q_negative_duration") | F.col("q_zero_duration"), "non_positive_duration")
                             .otherwise("duration_not_eligible")))
    return d.filter(F.col("eligible_duration")), rejected

In [ ]:
def reconcile_silver(bronze, silver, rejected):
    reconcile_order_keys(bronze, silver.select("order_id").unionByName(rejected.select("order_id")),
                         "Bronze → Silver/Rejected")
    valid_timestamp = F.col("accept_ts").isNotNull() & F.col("delivery_ts").isNotNull()
    eligible = F.coalesce(valid_timestamp & (F.col("delivery_ts") > F.col("accept_ts")), F.lit(False))
    expected_reason = F.when(~valid_timestamp, "invalid_timestamp").otherwise("non_positive_duration")
    if silver.filter(~eligible).limit(1).count():
        raise ValueError("Silver chứa dòng có timestamp lỗi hoặc thời lượng không dương")
    if rejected.filter(eligible | ~F.col("reject_reason").eqNullSafe(expected_reason)).limit(1).count():
        raise ValueError("Rejected chứa dòng hợp lệ hoặc sai lý do từ chối")
    return {"bronze_rows": bronze.count(), "silver_rows": silver.count(), "rejected_rows": rejected.count()}

In [ ]:
def transform_silver(spark):
    df = spark.read.format("delta").load(BRONZE)
    require_unique_orders(df)
    final, rejected = split_silver(df)
    final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(SILVER)
    rejected.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(REJECTED)
    return reconcile_silver(df, spark.read.format("delta").load(SILVER),
                            spark.read.format("delta").load(REJECTED))

In [1]:
result = transform_silver(spark)
result

{'bronze_rows': 206431, 'silver_rows': 206062, 'rejected_rows': 369}

### 3. Đối chiếu kết quả

Trừ hai timestamp trên một dòng Silver rồi so với thời lượng đã ghi.
Xem số đơn theo từng lý do trong Rejected.

In [ ]:
from math import isclose

silver = spark.read.format("delta").load(SILVER)
rejected = spark.read.format("delta").load(REJECTED)
example = silver.select("order_id", "accept_ts", "delivery_ts", "delivery_minutes").first()
hand_minutes = (example.delivery_ts - example.accept_ts).total_seconds() / 60
assert isclose(hand_minutes, example.delivery_minutes, rel_tol=0, abs_tol=1e-9)
print(f"Đơn {example.order_id}  {example.delivery_ts} - {example.accept_ts}")
print(f"Tính tay {hand_minutes:.2f} phút, cột Silver {example.delivery_minutes:.2f} phút")
rejected.groupBy("reject_reason").count().show(truncate=False)